# LAB3 — Bài 3: MNIST với 10-fold cross-validation

**Sinh viên:** Trần Chí Nguyên — **MSSV:** 6451071052.

> Mỗi câu lập trình in họ tên và MSSV theo template. Chạy lần lượt từ trên xuống.

In [1]:
%matplotlib inline
HO_TEN = 'Trần Chí Nguyên'
MSSV = '6451071052'
def in_danh_tinh(): print(f'{HO_TEN} - {MSSV}')


## Câu a–c. Đọc, xem kích thước và chuẩn hóa MNIST

Dùng tập train/test chuẩn từ Keras, chuẩn hóa pixel.

In [2]:
in_danh_tinh()
import numpy as np
from tensorflow.keras.datasets import mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print('Train:', train_images.shape, train_labels.shape, '| Test:', test_images.shape, test_labels.shape)
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0

Trần Chí Nguyên - 6451071052


Train: (60000, 28, 28) (60000,) | Test: (10000, 28, 28) (10000,)


## Câu d–e. Định nghĩa mô hình và chạy 10-fold cross-validation

Mỗi fold huấn luyện 5 epoch, batch 32; lưu loss/accuracy và mô hình từng fold. Full MNIST có thể cần thời gian và dung lượng đáng kể.

In [3]:
in_danh_tinh()
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from sklearn.model_selection import KFold

def create_model():
    m = Sequential([Flatten(input_shape=(28,28)), Dense(128, activation='relu'),
                    Dropout(0.2), Dense(10, activation='softmax')])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
models, accuracy_per_fold, loss_per_fold = [], [], []
model_dir = Path('outputs/mnist_kfold'); model_dir.mkdir(parents=True, exist_ok=True)
for fold, (train_idx, val_idx) in enumerate(kfold.split(train_images), start=1):
    print(f'\nFold {fold}/10')
    model = create_model()
    model.fit(train_images[train_idx], train_labels[train_idx],
              validation_data=(train_images[val_idx], train_labels[val_idx]),
              epochs=5, batch_size=32, verbose=1)
    loss, accuracy = model.evaluate(train_images[val_idx], train_labels[val_idx], verbose=0)
    print(f'Fold {fold}: loss={loss:.4f}, accuracy={accuracy:.4f}')
    models.append(model)
    loss_per_fold.append(float(loss)); accuracy_per_fold.append(float(accuracy))
    model.save(model_dir / f'mnist_fold_{fold}.keras')

Trần Chí Nguyên - 6451071052



Fold 1/10


C:\Users\Nguyen\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5


   1/1688 ━━━━━━━━━━━━━━━━━━━━ 2:51:02 6s/step - accuracy: 0.1250 - loss: 2.3808

   8/1688 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.1914 - loss: 2.2006   

... (9,940 outputs left out)

1676/1688 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9748 - loss: 0.0795

1683/1688 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9748 - loss: 0.0795

1688/1688 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9749 - loss: 0.0795 - val_accuracy: 0.9735 - val_loss: 0.0840


Fold 10: loss=0.0840, accuracy=0.9735


## Câu f–g. Thống kê folds và vẽ biểu đồ

Tính mean/std của accuracy và loss, vẽ hai chuỗi theo fold.

In [4]:
in_danh_tinh()
import matplotlib.pyplot as plt
print(f'Mean accuracy: {np.mean(accuracy_per_fold):.4f} ± {np.std(accuracy_per_fold):.4f}')
print(f'Mean loss: {np.mean(loss_per_fold):.4f} ± {np.std(loss_per_fold):.4f}')
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].plot(range(1,11),accuracy_per_fold,marker='o'); axes[0].set_title('Accuracy per fold'); axes[0].set_xlabel('Fold')
axes[1].plot(range(1,11),loss_per_fold,marker='o'); axes[1].set_title('Loss per fold'); axes[1].set_xlabel('Fold')
plt.tight_layout(); plt.show()

Trần Chí Nguyên - 6451071052
Mean accuracy: 0.9738 ± 0.0016
Mean loss: 0.0873 ± 0.0060


  plt.tight_layout(); plt.show()


## Câu h–k. Chọn mô hình tốt nhất và đánh giá tập test

Chọn fold accuracy cao nhất, dự đoán test, in accuracy, classification report và confusion matrix.

In [5]:
in_danh_tinh()
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
best_idx = int(np.argmax(accuracy_per_fold))
best_model = models[best_idx]
print('Fold tốt nhất:', best_idx+1, '| validation accuracy:', accuracy_per_fold[best_idx])
y_pred = np.argmax(best_model.predict(test_images, batch_size=256, verbose=0), axis=1)
print('Test accuracy:', accuracy_score(test_labels,y_pred))
print(classification_report(test_labels,y_pred,digits=4,zero_division=0))
sns.heatmap(confusion_matrix(test_labels,y_pred), cmap='Blues'); plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.show()

Trần Chí Nguyên - 6451071052


Fold tốt nhất: 1 | validation accuracy: 0.9763333201408386


Test accuracy: 0.976
              precision    recall  f1-score   support

           0     0.9838    0.9918    0.9878       980
           1     0.9860    0.9903    0.9881      1135
           2     0.9741    0.9845    0.9793      1032
           3     0.9457    0.9832    0.9641      1010
           4     0.9766    0.9766    0.9766       982
           5     0.9817    0.9596    0.9705       892
           6     0.9812    0.9802    0.9807       958
           7     0.9738    0.9747    0.9742      1028
           8     0.9810    0.9548    0.9677       974
           9     0.9778    0.9604    0.9690      1009

    accuracy                         0.9760     10000
   macro avg     0.9762    0.9756    0.9758     10000
weighted avg     0.9761    0.9760    0.9760     10000



  sns.heatmap(confusion_matrix(test_labels,y_pred), cmap='Blues'); plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.show()


## Câu l. Hiển thị dự đoán năm ảnh đầu

Hiển thị ảnh test, nhãn thật và nhãn dự đoán.

In [6]:
in_danh_tinh()
fig,axes=plt.subplots(1,5,figsize=(12,3))
for i,ax in enumerate(axes):
    ax.imshow(test_images[i],cmap='gray'); ax.set_title(f'Thật {test_labels[i]} / Dự đoán {y_pred[i]}'); ax.axis('off')
plt.tight_layout(); plt.show()

Trần Chí Nguyên - 6451071052


  plt.tight_layout(); plt.show()
